In [1]:
# Cell 1 — Imports & version check
import numpy as np
import pandas as pd
import scipy.optimize as sco
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
from datetime import datetime

print("All imports successful")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

from pathlib import Path
# Project folder, whether this runs from notebooks/ or the project root
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()


All imports successful
NumPy: 2.4.4
Pandas: 3.0.2


In [2]:
# Cell 2 — Download historical price data
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'GS', 'NVDA', 'TSLA', 'BRK-B', 'SPY']

start_date = '2020-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')

print(f"Downloading data from {start_date} to {end_date}...")

raw = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)
prices = raw['Close']

print(f"\nShape: {prices.shape}")
print(f"Tickers: {list(prices.columns)}")
print(f"\nFirst 3 rows:")
prices.head(3)

[*********************100%***********************]  10 of 10 completed


Shape: (1576, 10)
Tickers: ['AAPL', 'AMZN', 'BRK-B', 'GOOGL', 'GS', 'JPM', 'MSFT', 'NVDA', 'SPY', 'TSLA']

First 3 rows:


Ticker,AAPL,AMZN,BRK-B,GOOGL,GS,JPM,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,,,
2020-01-02,72.400513,94.900497,228.389999,67.873024,202.118927,118.430321,152.158371,5.970756,296.888245,28.684000
2020-01-03,71.696640,93.748497,226.179993,67.517952,199.755463,116.867470,150.263779,5.875186,294.640015,29.534000
2020-01-06,72.267929,95.143997,226.990005,69.317589,201.799728,116.774536,150.652161,5.899826,295.764099,30.102667


In [3]:
# Cell 3 — Compute daily returns & key statistics
returns = prices.pct_change().dropna()

# Annualized metrics
ann_return = returns.mean() * 252
ann_volatility = returns.std() * np.sqrt(252)
sharpe = ann_return / ann_volatility

stats = pd.DataFrame({
    'Annual Return (%)': (ann_return * 100).round(2),
    'Annual Volatility (%)': (ann_volatility * 100).round(2),
    'Sharpe Ratio': sharpe.round(3)
}).sort_values('Sharpe Ratio', ascending=False)

print(stats)

        Annual Return (%)  Annual Volatility (%)  Sharpe Ratio
Ticker                                                        
NVDA                69.01                  52.55         1.313
TSLA                61.41                  65.58         0.936
GOOGL               29.88                  32.25         0.927
GS                  29.44                  32.90         0.895
AAPL                25.45                  31.53         0.807
SPY                 15.36                  20.52         0.749
BRK-B               14.11                  21.09         0.669
JPM                 20.23                  31.16         0.649
MSFT                18.66                  29.66         0.629
AMZN                21.06                  35.59         0.592


In [4]:
# Cell 4 — Monte Carlo simulation: 10,000 random portfolios
np.random.seed(42)
n_assets = len(tickers)
n_simulations = 10_000

# Store results
mc_weights = np.zeros((n_simulations, n_assets))
mc_returns = np.zeros(n_simulations)
mc_volatility = np.zeros(n_simulations)
mc_sharpe = np.zeros(n_simulations)

# Covariance matrix (annualized)
cov_matrix = returns.cov() * 252

for i in range(n_simulations):
    # Random weights that sum to 1
    w = np.random.dirichlet(np.ones(n_assets))
    mc_weights[i] = w
    
    port_return = np.dot(w, ann_return)
    port_vol = np.sqrt(w.T @ cov_matrix.values @ w)
    
    mc_returns[i] = port_return
    mc_volatility[i] = port_vol
    mc_sharpe[i] = port_return / port_vol

print(f"Simulations complete: {n_simulations:,} portfolios")
print(f"\nBest Sharpe Ratio found: {mc_sharpe.max():.4f}")
print(f"Max Return found:        {mc_returns.max()*100:.2f}%")
print(f"Min Volatility found:    {mc_volatility.min()*100:.2f}%")

Simulations complete: 10,000 portfolios

Best Sharpe Ratio found: 1.3684
Max Return found:        53.70%
Min Volatility found:    20.77%


In [5]:
# Cell 5 — Plot the Efficient Frontier
# Find special portfolios
max_sharpe_idx = mc_sharpe.argmax()
min_vol_idx = mc_volatility.argmin()

fig = go.Figure()

# All simulated portfolios
fig.add_trace(go.Scatter(
    x=mc_volatility * 100,
    y=mc_returns * 100,
    mode='markers',
    marker=dict(
        color=mc_sharpe,
        colorscale='Viridis',
        size=3,
        opacity=0.6,
        colorbar=dict(title='Sharpe Ratio')
    ),
    name='Simulated Portfolios',
    text=[f"Sharpe: {s:.3f}" for s in mc_sharpe],
    hovertemplate='Vol: %{x:.2f}%<br>Return: %{y:.2f}%<br>%{text}'
))

# Max Sharpe portfolio
fig.add_trace(go.Scatter(
    x=[mc_volatility[max_sharpe_idx] * 100],
    y=[mc_returns[max_sharpe_idx] * 100],
    mode='markers',
    marker=dict(color='red', size=15, symbol='star'),
    name=f'Max Sharpe ({mc_sharpe[max_sharpe_idx]:.3f})'
))

# Min Volatility portfolio
fig.add_trace(go.Scatter(
    x=[mc_volatility[min_vol_idx] * 100],
    y=[mc_returns[min_vol_idx] * 100],
    mode='markers',
    marker=dict(color='green', size=15, symbol='diamond'),
    name=f'Min Volatility ({mc_volatility[min_vol_idx]*100:.2f}%)'
))

fig.update_layout(
    title='Efficient Frontier — 10,000 Monte Carlo Simulated Portfolios',
    xaxis_title='Annual Volatility (%)',
    yaxis_title='Annual Return (%)',
    template='plotly_dark',
    width=900,
    height=600,
    legend=dict(x=0.01, y=0.99)
)

fig.show()

In [6]:
# Cell 6 — Optimize: Max Sharpe & Min Volatility portfolios
from scipy.optimize import minimize

def neg_sharpe(weights):
    r = np.dot(weights, ann_return)
    v = np.sqrt(weights.T @ cov_matrix.values @ weights)
    return -r / v

def portfolio_vol(weights):
    return np.sqrt(weights.T @ cov_matrix.values @ weights)

# Constraints: weights sum to 1
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
# Bounds: each weight between 0% and 40%
bounds = tuple((0.0, 0.40) for _ in range(n_assets))
# Starting guess: equal weight
w0 = np.ones(n_assets) / n_assets

# Max Sharpe optimization
opt_sharpe = minimize(neg_sharpe, w0, method='SLSQP',
                      bounds=bounds, constraints=constraints)

# Min Volatility optimization
opt_vol = minimize(portfolio_vol, w0, method='SLSQP',
                   bounds=bounds, constraints=constraints)

# Display results
def show_portfolio(label, weights, tickers):
    r = np.dot(weights, ann_return)
    v = np.sqrt(weights.T @ cov_matrix.values @ weights)
    s = r / v
    print(f"\n{'='*45}")
    print(f"  {label}")
    print(f"{'='*45}")
    print(f"  Annual Return:     {r*100:.2f}%")
    print(f"  Annual Volatility: {v*100:.2f}%")
    print(f"  Sharpe Ratio:      {s:.4f}")
    print(f"\n  Weights:")
    for t, w in sorted(zip(tickers, weights), key=lambda x: -x[1]):
        if w > 0.001:
            print(f"    {t:<8} {w*100:.2f}%")

show_portfolio("Max Sharpe Portfolio", opt_sharpe.x, tickers)
show_portfolio("Min Volatility Portfolio", opt_vol.x, tickers)


  Max Sharpe Portfolio
  Annual Return:     48.25%
  Annual Volatility: 34.68%
  Sharpe Ratio:      1.3912

  Weights:
    TSLA     40.00%
    JPM      25.68%
    SPY      13.23%
    AMZN     12.59%
    GOOGL    8.49%

  Min Volatility Portfolio
  Annual Return:     16.20%
  Annual Volatility: 19.85%
  Sharpe Ratio:      0.8162

  Weights:
    GOOGL    40.00%
    BRK-B    40.00%
    NVDA     8.48%
    MSFT     5.37%
    AMZN     4.75%
    GS       1.40%


In [7]:
# Cell 7 — Black-Scholes-Merton Options Pricing
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    """
    S     = current stock price
    K     = strike price
    T     = time to expiration (years)
    r     = risk-free rate
    sigma = annual volatility
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = norm.cdf(d1)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = -norm.cdf(-d1)

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100

    return {
        'Price': round(price, 4),
        'Delta': round(delta, 4),
        'Gamma': round(gamma, 6),
        'Theta': round(theta, 4),
        'Vega':  round(vega, 4)
    }

# Price options on all 10 tickers using current price + implied vol
print(f"{'Ticker':<8} {'Price':>8} {'Strike':>8} {'Call $':>8} {'Put $':>8} {'Delta':>8} {'Vega':>8}")
print("-" * 60)

r_f = 0.053  # current risk-free rate (~5.3% US 10Y)
T   = 0.25   # 3 months to expiration

for ticker in tickers:
    S     = prices[ticker].iloc[-1]
    K     = round(S, 0)  # at-the-money
    sigma = returns[ticker].std() * np.sqrt(252)

    call = black_scholes(S, K, T, r_f, sigma, 'call')
    put  = black_scholes(S, K, T, r_f, sigma, 'put')

    print(f"{ticker:<8} {S:>8.2f} {K:>8.2f} {call['Price']:>8.4f} "
          f"{put['Price']:>8.4f} {call['Delta']:>8.4f} {call['Vega']:>8.4f}")

Ticker      Price   Strike   Call $    Put $    Delta     Vega
------------------------------------------------------------
AAPL       260.48   260.00  18.2672  14.3650   0.5693   0.5117
MSFT       370.87   371.00  24.2382  19.4849   0.5640   0.7302
GOOGL      317.24   317.00  22.5248  18.1122   0.5665   0.6240
AMZN       238.38   238.00  18.5915  15.0788   0.5684   0.4685
JPM        309.87   310.00  21.1564  17.2060   0.5637   0.6102
GS         907.80   908.00  65.1760  53.4244   0.5641   1.7873
NVDA       188.63   189.00  20.6827  18.5650   0.5692   0.3706
TSLA       348.95   349.00  47.4586  42.9148   0.5808   0.6817
BRK-B      479.90   480.00  23.3139  17.0959   0.5700   0.9425
SPY        679.46   679.00  32.5567  23.1592   0.5742   1.3318


In [8]:
# Cell 8 — Greeks Sensitivity (Volatility Smile)
sigmas = np.linspace(0.10, 0.80, 50)
strikes = np.linspace(0.7, 1.3, 50)  # as % of spot

S = prices['AAPL'].iloc[-1]
T = 0.25
r_f = 0.053

call_prices = []
for k_pct in strikes:
    K = S * k_pct
    sigma = returns['AAPL'].std() * np.sqrt(252)
    result = black_scholes(S, K, T, r_f, sigma, 'call')
    call_prices.append(result['Price'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=strikes * 100,
    y=call_prices,
    mode='lines+markers',
    line=dict(color='royalblue', width=2),
    name='AAPL Call Price'
))
fig.update_layout(
    title='AAPL Call Option Price vs Strike (% of Spot)',
    xaxis_title='Strike as % of Spot Price',
    yaxis_title='Call Option Price ($)',
    template='plotly_dark',
    width=800, height=450
)
fig.show()

In [9]:
# Cell 9 — Portfolio Weights Visualization
labels = list(tickers)
max_sharpe_weights = opt_sharpe.x
min_vol_weights = opt_vol.x

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Max Sharpe',
    x=labels,
    y=(max_sharpe_weights * 100).round(2),
    marker_color='crimson'
))
fig.add_trace(go.Bar(
    name='Min Volatility',
    x=labels,
    y=(min_vol_weights * 100).round(2),
    marker_color='seagreen'
))
fig.update_layout(
    title='Optimal Portfolio Weights — Max Sharpe vs Min Volatility',
    xaxis_title='Asset',
    yaxis_title='Weight (%)',
    barmode='group',
    template='plotly_dark',
    width=850, height=480
)
fig.show()

In [10]:
# Cell 10 — Monte Carlo Future Price Paths (GBM)
# This connects directly to your Sandisk/ASML project
n_days = 252        # 1 trading year
n_paths = 200       # paths per asset
dt = 1/252

fig = go.Figure()

for ticker in ['AAPL', 'MSFT', 'NVDA', 'JPM']:
    S0    = prices[ticker].iloc[-1]
    mu    = ann_return[ticker]
    sigma = returns[ticker].std() * np.sqrt(252)

    paths = np.zeros((n_days, n_paths))
    paths[0] = S0

    for t in range(1, n_days):
        Z = np.random.standard_normal(n_paths)
        paths[t] = paths[t-1] * np.exp(
            (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
        )

    for j in range(n_paths):
        fig.add_trace(go.Scatter(
            x=list(range(n_days)),
            y=paths[:, j],
            mode='lines',
            line=dict(width=0.5),
            opacity=0.3,
            showlegend=False,
            name=ticker
        ))

    # Mean path
    fig.add_trace(go.Scatter(
        x=list(range(n_days)),
        y=paths.mean(axis=1),
        mode='lines',
        line=dict(width=2.5),
        name=f'{ticker} mean path'
    ))

fig.update_layout(
    title='GBM Monte Carlo Price Paths — 200 Simulations per Asset',
    xaxis_title='Trading Days',
    yaxis_title='Simulated Price ($)',
    template='plotly_dark',
    width=900, height=550
)
fig.show()

In [11]:
# Cell 11 — Export results to outputs folder
import os
import json

output_dir = str(PROJECT_DIR / 'outputs')
os.makedirs(output_dir, exist_ok=True)

# Save stats table
stats.to_csv(os.path.join(output_dir, 'asset_statistics.csv'))

# Save optimal weights
weights_df = pd.DataFrame({
    'Ticker': tickers,
    'Max Sharpe Weight (%)': (opt_sharpe.x * 100).round(2),
    'Min Vol Weight (%)': (opt_vol.x * 100).round(2)
})
weights_df.to_csv(os.path.join(output_dir, 'optimal_weights.csv'), index=False)

# Save summary JSON for website
summary = {
    'max_sharpe': {
        'sharpe': round(float(-opt_sharpe.fun), 4),
        'return_pct': round(float(np.dot(opt_sharpe.x, ann_return)) * 100, 2),
        'volatility_pct': round(float(np.sqrt(opt_sharpe.x @ cov_matrix.values @ opt_sharpe.x)) * 100, 2)
    },
    'min_vol': {
        'volatility_pct': round(float(np.sqrt(opt_vol.x @ cov_matrix.values @ opt_vol.x)) * 100, 2),
        'return_pct': round(float(np.dot(opt_vol.x, ann_return)) * 100, 2)
    }
}

with open(os.path.join(output_dir, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("Exports saved:")
for f in os.listdir(output_dir):
    print(f"  {f}")

Exports saved:
  asset_statistics.csv
  optimal_weights.csv
  summary.json
